# E2E Part 4 — optional SURT training and final evaluation

This notebook implements Sections 10–12 of `docx/steps.md`. Phase 8 training is optional and cannot start until a version-pinned external Icefall/SURT training backend is integrated. Phase 9 is the final aggregation stage and requires comparable per-example `EvaluationRecord` JSONL files.

In [ ]:
%cd /kaggle/working/English-Streaming-ASR
from pathlib import Path
ROOT = Path.cwd()
assert (ROOT / 'src').is_dir()

# 10. Optional Phase 8 — training-contract validation

This validates split manifests and target construction only. It does not train a model without a real external `model.backend_factory`.

In [ ]:
from pathlib import Path

Path('kaggle_training.yaml').write_text('''\
dataset:
  train_manifest: /kaggle/working/English-Streaming-ASR/outputs/synthetic_overlap_dev/manifests/train.jsonl
  validation_manifest: /kaggle/working/English-Streaming-ASR/outputs/synthetic_overlap_dev/manifests/validation.jsonl
  test_manifest: /kaggle/working/English-Streaming-ASR/outputs/synthetic_overlap_dev/manifests/test.jsonl
''')
print(Path('kaggle_training.yaml').read_text())

!python scripts/train_asr.py \
  --config configs/training.yaml \
  --override kaggle_training.yaml \
  --validate-only

A real train/resume run additionally needs a pinned external Icefall/SURT recipe, checkpoint, tokenizer/BPE assets, model backend factory, and documented checkpoint format. Persist model/adapter weights, optimizer, scheduler, trainer state, epoch, global step, random state, configs, manifests, and tokenizer assets.

# 11. Phase 9 — freeze the final evaluation

In [ ]:
from pathlib import Path

# Create this only after choosing final checkpoints and attaching persistent datasets.
Path('kaggle_evaluation.yaml').write_text('''\
frozen_system:
  whisperrt_checkpoint: /kaggle/input/<your-whisperrt-dataset>/small_300.pt
  surt_checkpoint: /kaggle/input/<your-surt-checkpoint-dataset>/<checkpoint-file>
  finetuned_surt_checkpoint: null
  train_manifest: /kaggle/input/<your-overlap-dataset>/manifests/train.jsonl
  validation_manifest: /kaggle/input/<your-overlap-dataset>/manifests/validation.jsonl
  test_manifest: /kaggle/input/<your-overlap-dataset>/manifests/test.jsonl
''')
print(Path('kaggle_evaluation.yaml').read_text())

In [ ]:
# Run after replacing the placeholder paths.
# !python scripts/evaluate.py \
#   --config configs/evaluation.yaml \
#   --override kaggle_evaluation.yaml \
#   --validate-only

## 11.3 Aggregate actual comparable records

`scripts/evaluate.py` requires one or more files in the project's `EvaluationRecord` JSONL schema. Do not pass a Phase 4 result file or routing trace until it has been converted/verified against that schema. This repository intentionally has no automatic conversion that could mislabel incompatible result types.

In [ ]:
# Replace the path with a real, schema-valid consolidated EvaluationRecord JSONL file.
# !python scripts/evaluate.py \
#   --config configs/evaluation.yaml \
#   --override kaggle_evaluation.yaml \
#   --records /kaggle/working/outputs/predictions/final_records.jsonl
#
# !python scripts/benchmark.py \
#   --records /kaggle/working/outputs/predictions/final_records.jsonl

# 12. Final delivery checklist

Before making an E2E claim, create a versioned private Kaggle Dataset containing: all base/override configs; train/validation/test manifests and hashes; permitted checkpoints; per-example prediction records; raw metrics; Phase 1/4/5/6/7/9 reports; `pip freeze`/GPU/Python/CUDA details; Git commit; and a short README with command order and limitations.

The final comparison must include clean WhisperRT, WhisperRT under overlap, adaptive `always_normal`, adaptive `predicted`, adaptive `oracle` (marked non-deployable), direct SURT, and adaptive WhisperRT + SURT. Include fine-tuned SURT only after a reproducible external training run.